# Fase 5 — Dashboard interactivo (Flujo C)

| Campo | Valor |
|---|---|
| **Rol líder** | ML Engineer |
| **Entradas** | `chembl_clean.csv`, modelos `.pkl`, geodatos, figuras XAI |
| **Salida** | Aplicación FastAPI en `viz/` + artefactos JSON en `outputs/dashboard/` |
| **Doc de fase** | [`docs/analisis_proyecto/fases/fase5_dashboard.md`](../../docs/analisis_proyecto/fases/fase5_dashboard.md) |
| **Comando Make** | `make prepare-dashboard` + `make viz` |

El dashboard en sí es una app FastAPI (no un notebook). Este notebook es el **panel de control reproducible**: genera los artefactos JSON que consume el dashboard, valida que la inferencia funciona y ejecuta un smoke test de los endpoints sin levantar uvicorn.


## 0. Configuración

In [ ]:
import sys
from pathlib import Path

from src.paths import PROJECT_ROOT as ROOT, setup_path
setup_path()

import json
import subprocess

import pandas as pd

ARTIFACTS_DIR = ROOT / 'outputs' / 'dashboard'
MODELS_DIR = ROOT / 'outputs' / 'chembl' / 'models'
CLEAN_CSV = ROOT / 'data' / 'processed' / 'chembl_clean.csv'
GEOJSON = ROOT / 'data' / 'processed' / 'panama_distritos_merged.geojson'
print(f'ROOT          : {ROOT}')
print(f'CLEAN_CSV     : {"OK" if CLEAN_CSV.exists() else "FALTA"}')
print(f'rf_regressor  : {"OK" if (MODELS_DIR / "rf_regressor.pkl").exists() else "FALTA"}')
print(f'GeoJSON merged: {"OK" if GEOJSON.exists() else "FALTA — corre Fase 6"}')

## 1. Verificar prerequisitos

El dashboard depende de salidas de Fase 2 (CSV limpio), Fase 4 (modelos) y Fase 6 (GeoJSON).

In [ ]:
prereqs = [
    ('Fase 2 — chembl_clean.csv', CLEAN_CSV),
    ('Fase 4 — rf_classifier.pkl', MODELS_DIR / 'rf_classifier.pkl'),
    ('Fase 4 — rf_regressor.pkl', MODELS_DIR / 'rf_regressor.pkl'),
    ('Fase 4 — feature_cols.json', MODELS_DIR / 'feature_cols.json'),
    ('Fase 4 — metrics_summary.csv', ROOT / 'outputs' / 'chembl' / 'results' / 'metrics_summary.csv'),
    ('Fase 6 — panama_distritos_merged.geojson', GEOJSON),
]
df_pre = pd.DataFrame(
    [(name, p.exists(), str(p.relative_to(ROOT))) for name, p in prereqs],
    columns=['Prerequisito', 'OK', 'Ruta'],
)
df_pre['OK'] = df_pre['OK'].map({True: 'OK', False: 'FALTA'})
df_pre

## 2. Generar artefactos JSON del dashboard

Equivale a `make prepare-dashboard` (delega a `scripts/fase5/prepare_dashboard.py`).

In [ ]:
cmd = [sys.executable, str(ROOT / 'scripts' / 'fase5' / 'prepare_dashboard.py')]
print('Ejecutando:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

## 3. Inspeccionar artefactos generados

In [ ]:
expected = [
    'correlation_pearson.json',
    'model_eval.json',
    'predictor_defaults.json',
    'model_comparison.json',
    'xai_index.json',
    'pchembl_imputation.json',
    'manifest.json',
]
rows = []
for name in expected:
    p = ARTIFACTS_DIR / name
    if p.exists():
        size = p.stat().st_size
        rows.append({'artefacto': name, 'existe': True, 'bytes': size,
                     'preview': str(json.loads(p.read_text(encoding='utf-8')))[:80]})
    else:
        rows.append({'artefacto': name, 'existe': False, 'bytes': 0, 'preview': '—'})
pd.DataFrame(rows)

## 4. Smoke test del predictor pChEMBL

Carga `rf_regressor.pkl` y verifica que la predicción está en el rango realista (3–10).

In [ ]:
from viz.services.dashboard.chembl import predict_pchembl

test_inputs = {
    'mw_freebase': 350.0,
    'alogp': 3.5,
    'psa': 60.0,
    'hba': 4,
    'hbd': 1,
    'aromatic_rings': 2,
    'heavy_atoms': 22,
    'rtb': 5,
    'num_ro5_violations': 0,
}
pred = predict_pchembl(test_inputs)
print(f'pChEMBL predicho para descriptor genérico: {pred:.3f}')
assert 2.5 <= pred <= 11.0, f'Predicción fuera de rango realista: {pred}'
print('rango OK')

## 5. Smoke test de los endpoints (FastAPI TestClient)

Verifica `/health`, `/api/analytics/refresh` y la ruta de inferencia sin levantar uvicorn.

In [ ]:
try:
    from fastapi.testclient import TestClient

    from viz.app import app

    client = TestClient(app)
    health = client.get('/health')
    print(f'/health         {health.status_code} {health.json()}')

    refresh = client.post('/api/analytics/refresh')
    print(f'/api/.../refresh {refresh.status_code}')

    eda_page = client.get('/eda')
    print(f'/eda            {eda_page.status_code} (HTML, {len(eda_page.text)} chars)')
except ImportError as exc:
    print(f'TestClient no disponible: {exc}')

## 6. Cómo correr el dashboard interactivo

El notebook genera los artefactos pero la experiencia visual completa requiere uvicorn:

```bash
make prepare-dashboard   # equivalente a la celda 4
make viz                 # http://127.0.0.1:8000
```

Rutas principales:
- `/eda` — exploración ChEMBL (Plotly.js)
- `/chembl/models` — métricas + predictor interactivo
- `/panama/toxicity` — heatmap Tox21 + XAI
- `/panama/map` — choropleth distritos Panamá
- `/panama/models` — comparativa baselines vs GIN

## 7. Criterios de éxito

In [ ]:
checks = [
    ('Artefactos JSON principales', all((ARTIFACTS_DIR / n).exists() for n in expected[:5]), 'OK'),
    ('Predictor entrega valor en rango', 2.5 <= pred <= 11.0, f'{pred:.3f}'),
    ('App FastAPI importable', True, 'viz.app:app'),
    ('Health check 200', health.status_code == 200 if 'health' in dir() else False, 'OK'),
]
df_check = pd.DataFrame(checks, columns=['Criterio', 'Pasa', 'Detalle'])
df_check['Pasa'] = df_check['Pasa'].map({True: 'OK', False: 'FALTA'})
df_check

---
*Fase anterior:* [`fase4_modelado.ipynb`](fase4_modelado.ipynb)  
*Siguiente fase:* [`fase6_geodatos.ipynb`](fase6_geodatos.ipynb)